In [1]:
import os
os.chdir('../../../../..')

In [2]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
from pyriemann.tangentspace import TangentSpace
import umap
import seaborn as sns
from loguru import logger
from sklearn.manifold import TSNE, MDS

from src.datasets import QM9Dataset
from src.non_euclidean import Riemann
from src.helper_functions import create_chemiscope_viewer

In [3]:
qm9 = QM9Dataset(limit=80_000, sampling_strategy="stratified", descriptors=["soap"])
df = qm9.load()

2026-05-23 16:13:13.639 | INFO     | src.datasets:_load_full_qm9_df:817 - Loading cached full QM9 dataset from: data/QM9/dataset_cleaned.parquet
2026-05-23 16:13:14.075 | INFO     | src.datasets:_sample_qm9_df:1000 - QM9 sampling complete: strategy=stratified, requested_limit=80000, returned_rows=80000, sampling on columns=['num_atoms', 'gap'].
2026-05-23 16:13:14.076 | INFO     | src.datasets:_add_requested_descriptors:202 - Applying requested QM9 descriptors to sampled dataframe (rows=80000).
2026-05-23 16:13:14.157 | INFO     | src.features:compute_soap_outputs:395 - Computing SOAP (rcut=6.0, nmax=8, lmax=6, normalize=True)...
2026-05-23 16:15:55.717 | SUCCESS  | src.datasets:add_soap:1193 - Added SOAP embeddings and matrices.
2026-05-23 16:15:55.782 | INFO     | src.datasets:_add_requested_descriptors:213 - Added descriptor column(s): ['soap_embedding', 'soap_matrix']
2026-05-23 16:15:55.887 | INFO     | src.datasets:_load_with_descriptor_filter:857 - QM9 descriptor null-filtering 

In [4]:
from dscribe.kernels import REMatchKernel

def rematch_kernel_dist_matrix(df, metric="linear", alpha=0.1, tol=1e-3):

    soap_matrices = df["soap_matrix"].to_list()

    cleaned = []

    normalized = False
    for i, x in enumerate(soap_matrices):
        x = np.asarray(x, dtype=np.float64)

        # shape check
        if x.ndim != 2:
            raise ValueError(f"SOAP matrix {i} is not 2D: shape={x.shape}")

        # finite check
        if not np.isfinite(x).all():
            raise ValueError(f"Non-finite values in SOAP matrix {i}")

        # ----------------------------
        # Check if already normalized
        # ----------------------------
        norms = np.linalg.norm(x, axis=1)

        already_normalized = np.all(np.abs(norms - 1.0) < tol)

        if already_normalized:
            x_norm = x
            normalized = True
            cleaned = soap_matrices
            logger.info("Descriptor is already normalized")
            if normalized:
                break
        else:
            x_norm = x / (norms[:, None] + 1e-12)
        

        # final sanity check
        if not np.isfinite(x_norm).all():
            raise ValueError(f"NaN/inf after normalization in matrix {i}")

        cleaned.append(x_norm)

    # ----------------------------
    # REMatch kernel
    # ----------------------------
    kernel = REMatchKernel(
        metric=metric,
        alpha=alpha
    )

    print("Computing REMatch kernel matrix...")
    K = kernel.create(cleaned)

    if not np.isfinite(K).all():
        raise ValueError("Kernel matrix contains NaN or inf")

    # ----------------------------
    # Distance conversion
    # ----------------------------
    diag = np.diag(K)
    dist_sq = diag[:, None] + diag[None, :] - 2.0 * K

    dist_sq = np.clip(dist_sq, 0.0, None)
    D = np.sqrt(dist_sq)

    np.fill_diagonal(D, 0.0)
    D = (D + D.T) / 2.0

    return D

# Hypothesis 1
- goal is to show that some molecules can not be well seperaeted by ReMatch kernel. These molecules could be highly structural/constitutional isomers with identical local chemical environments but radically different global topologies
- method: compare cyclic vs linear isomers. choose C6H10. 
- reason: the local hybridization will look very similar for rematch kernel. However, because the structural variations are constrained to a ring vs. stretched along a chain, the features will co-vary differently across the atoms, creating distinct covariance footprints

In [5]:
df = df.filter(pl.col("num_atoms") > 7) # works better on riemann

df_C5H8O = df.filter(pl.col("formula") == "C5H8O")
df_C7H10O2 = df.filter(pl.col("formula") == "C7H10O2").sample(500, with_replacement=True)
print(df_C7H10O2.shape)

(500, 72)


In [6]:
riemann = Riemann()
dist_matrix_riemann = riemann.distance_matrix(df_C7H10O2, "soap", distance_type='affine-invariant')
labels = [i for i in range(len(df_C7H10O2))]

2026-05-23 16:15:58.476 | INFO     | src.non_euclidean:distance_matrix:1113 - Computing Riemann distance matrix | Features: soap | Distance: affine-invariant
2026-05-23 16:15:58.477 | INFO     | src.non_euclidean:_feature_matrices_from_df:332 - Using column: soap_matrix from df
2026-05-23 16:15:58.581 | INFO     | src.non_euclidean:matrix_pca:1075 - Applying PCA to reduce feature dimension to 17...
2026-05-23 16:15:58.637 | INFO     | src.non_euclidean:matrix_pca:1090 - PCA explained variance ratio: 0.9722 (cumulative for 17 components)
2026-05-23 16:15:58.728 | INFO     | src.non_euclidean:distance_matrix:1123 - Computing affine-invariant distances...


In [7]:
dist_matrix_rematch = rematch_kernel_dist_matrix(df_C7H10O2)

2026-05-23 16:15:59.659 | INFO     | __main__:rematch_kernel_dist_matrix:32 - Descriptor is already normalized


Computing REMatch kernel matrix...


In [8]:
create_chemiscope_viewer(df_C7H10O2, dist_matrix_riemann, labels, 'UMAP')

2026-05-23 16:16:35.780 | INFO     | src.helper_functions:create_chemiscope_viewer:1163 - Running UMAP dimensionality reduction...
2026-05-23 16:16:35.781 | INFO     | src.helper_functions:create_chemiscope_viewer:1172 - Converting structures/molecules to ASE Atoms for Chemiscope...
2026-05-23 16:16:39.473 | INFO     | src.helper_functions:create_chemiscope_viewer:1212 - Projecting using UMAP with n_neighbours = 15
/Users/karlfindhansen/thesis/Anomaly-Detection-in-Molecular-and-Materials-Datasets/.venv/lib/python3.12/site-packages/umap/umap_.py:1865: UserWarning: using precomputed metric; inverse_transform will be unavailable
  warn("using precomputed metric; inverse_transform will be unavailable")
/Users/karlfindhansen/thesis/Anomaly-Detection-in-Molecular-and-Materials-Datasets/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
OMP: Info #276: omp_set_nested routine deprecate

<ChemiscopeWidget(meta={'name': 'QM9 - UMAP Clustering'}, settings={'map': {'x': {'property': 'UMAP_1'}, 'y': …

In [9]:
create_chemiscope_viewer(df_C7H10O2, dist_matrix_rematch, labels, 'UMAP')

2026-05-23 16:16:43.484 | INFO     | src.helper_functions:create_chemiscope_viewer:1163 - Running UMAP dimensionality reduction...
2026-05-23 16:16:43.484 | INFO     | src.helper_functions:create_chemiscope_viewer:1172 - Converting structures/molecules to ASE Atoms for Chemiscope...
2026-05-23 16:16:47.147 | INFO     | src.helper_functions:create_chemiscope_viewer:1212 - Projecting using UMAP with n_neighbours = 15
/Users/karlfindhansen/thesis/Anomaly-Detection-in-Molecular-and-Materials-Datasets/.venv/lib/python3.12/site-packages/umap/umap_.py:1865: UserWarning: using precomputed metric; inverse_transform will be unavailable
  warn("using precomputed metric; inverse_transform will be unavailable")
/Users/karlfindhansen/thesis/Anomaly-Detection-in-Molecular-and-Materials-Datasets/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
2026-05-23 16:16:47.543 | INFO     | src.helper_

<ChemiscopeWidget(meta={'name': 'QM9 - UMAP Clustering'}, settings={'map': {'x': {'property': 'UMAP_1'}, 'y': …

In [10]:
df.columns

['mol_id',
 'formula',
 'smiles',
 'canonical_smiles',
 'scaffold_smiles',
 'generic_scaffold',
 'root_scaffold',
 'brics_fragments',
 'scaffold_tree_nodes',
 'selfies',
 'functional_groups',
 'structure_class',
 'is_injected',
 'outlier_category',
 'mol_weight',
 'logp',
 'tpsa',
 'election_affinity',
 'ionization_energies',
 'num_heavy_atoms',
 'num_rings',
 'num_aromatic_rings',
 'num_fluorine',
 'num_heteroatoms',
 'num_atoms',
 'coordination',
 'num_rotatable_bonds',
 'fraction_csp1',
 'fraction_csp2',
 'fraction_csp3',
 'h_bond_donors',
 'h_bond_acceptors',
 'branching_index',
 'num_sp_carbons',
 'num_sp2_carbons',
 'num_sp3_carbons',
 'main_chain_length',
 'raw_token_count',
 'avg_bond_length',
 'fr_benzene',
 'fr_alcohol',
 'fr_phenol',
 'fr_amine',
 'fr_amide',
 'fr_carboxylic_acid',
 'fr_ester',
 'fr_ketone',
 'fr_ether',
 'fr_nitro',
 'coordinates',
 'atomic_numbers',
 'mu',
 'alpha',
 'homo',
 'lumo',
 'gap',
 'r2',
 'zpve',
 'u0',
 'u',
 'h',
 'g',
 'cv',
 'u0_atom',
 'u_a

In [11]:
import numpy as np
import polars as pl
from scipy.stats import spearmanr
from sklearn.metrics import silhouette_score

riemann = Riemann()
dist_matrix_riemann_log = riemann.distance_matrix(
    df_C7H10O2, "soap", distance_type="log-euclidean"
)

dist_matrix_riemann_affine = riemann.distance_matrix(
    df_C7H10O2, "soap", distance_type="affine-invariant"
)

dist_matrix_rematch = rematch_kernel_dist_matrix(df_C7H10O2)

2026-05-23 16:16:47.604 | INFO     | src.non_euclidean:distance_matrix:1113 - Computing Riemann distance matrix | Features: soap | Distance: log-euclidean
2026-05-23 16:16:47.604 | INFO     | src.non_euclidean:_feature_matrices_from_df:332 - Using column: soap_matrix from df
2026-05-23 16:16:47.730 | INFO     | src.non_euclidean:matrix_pca:1075 - Applying PCA to reduce feature dimension to 17...
2026-05-23 16:16:47.742 | INFO     | src.non_euclidean:matrix_pca:1090 - PCA explained variance ratio: 0.9722 (cumulative for 17 components)
2026-05-23 16:16:47.810 | INFO     | src.non_euclidean:distance_matrix:1123 - Computing log-euclidean distances...
2026-05-23 16:16:47.819 | INFO     | src.non_euclidean:distance_matrix:1113 - Computing Riemann distance matrix | Features: soap | Distance: affine-invariant
2026-05-23 16:16:47.819 | INFO     | src.non_euclidean:_feature_matrices_from_df:332 - Using column: soap_matrix from df
2026-05-23 16:16:47.899 | INFO     | src.non_euclidean:matrix_pca:

Computing REMatch kernel matrix...


In [12]:


def run_topology_benchmark(
    df: pl.DataFrame, D_rematch: np.ndarray, D_riemann: np.ndarray
) -> None:
    """Evaluates how well REMatch vs Riemann matrices capture global shape

    changes using the exact rows present in the provided dataframe.
    """
    N = df.height

    # Safety check to ensure matrices match the dataframe dimensions
    if D_rematch.shape != (N, N) or D_riemann.shape != (N, N):
        raise ValueError(
            f"Dimension mismatch! DataFrame has {N} rows, "
            f"but REMatch is {D_rematch.shape} and Riemann is {D_riemann.shape}."
        )

    # Extract upper triangles for pair-wise correlation (excludes diagonal 0s)
    triu_idx = np.triu_indices(N, k=1)
    flat_rematch = D_rematch[triu_idx]
    flat_riemann = D_riemann[triu_idx]

    print(f"Running benchmark on {N} aligned molecules.\n")

    # ANALYSIS A: CORRELATION WITH MACRO-GEOMETRY
    print("=== Analysis A: Global Shape Correlation (Spearman ρ) ===")
    print(
        f"{'QM9 Feature':<20} | {'REMatch Correlation':<20} | {'Riemann Correlation':<20}"
    )
    print("-" * 68)

    # Updated features list to include electronic, thermodynamic, and quantum properties
    #shape_features = ["A", "B", "C", "r2", "alpha", "cv", "gap", "logp", "tpsa","u0", "mu"]
    shape_features = ["mu", "alpha", "homo", "lumo", "gap", "r2", "zpve", "u0", 
        "u", "h", "g", "cv", "u0_atom", "u_atom", "h_atom", "g_atom", 
        "A", "B", "C"]
    for feat in shape_features:
        if feat in df.columns:
            feat_vals = df.get_column(feat).to_numpy()
            # Pairwise absolute differences |x_i - x_j|
            D_feat = np.abs(feat_vals[:, None] - feat_vals[None, :])
            flat_feat = D_feat[triu_idx]

            corr_rematch, _ = spearmanr(flat_rematch, flat_feat)
            corr_riemann, _ = spearmanr(flat_riemann, flat_feat)

            print(
                f"{feat:<20} | {corr_rematch:<19.4f} | {corr_riemann:<19.4f}"
            )

    # ANALYSIS B: SEPARATION OF TOPOLOGICAL CLASSES
    print("\n=== Analysis B: Clustering Performance (Silhouette Score) ===")
    target_classes = ["structure_class", "num_rings"]

    for target_col in target_classes:
        if target_col in df.columns:
            labels = df.get_column(target_col).to_numpy()

            # Silhouette score requires at least 2 unique classes, and fewer than N classes
            unique_classes = len(np.unique(labels))
            if 1 < unique_classes < N:
                sil_rematch = silhouette_score(
                    D_rematch, labels, metric="precomputed"
                )
                sil_riemann = silhouette_score(
                    D_riemann, labels, metric="precomputed"
                )
                print(f"Target Class: {target_col}")
                print(f"  -> REMatch Silhouette Score : {sil_rematch:.4f}")
                print(f"  -> Riemann Silhouette Score : {sil_riemann:.4f}")
            else:
                print(
                    f"Target Class '{target_col}' skipped: requires between 2 and {N-1} unique classes (found {unique_classes})."
                )


# =====================================================================
# 3. Execute
# =====================================================================
run_topology_benchmark(
    df=df_C7H10O2, D_rematch=dist_matrix_rematch, D_riemann=dist_matrix_riemann_log
)

Running benchmark on 500 aligned molecules.

=== Analysis A: Global Shape Correlation (Spearman ρ) ===
QM9 Feature          | REMatch Correlation  | Riemann Correlation 
--------------------------------------------------------------------
mu                   | -0.0246             | -0.0087            
alpha                | 0.5493              | 0.4756             
homo                 | 0.0677              | 0.0258             
lumo                 | 0.0497              | 0.0490             
gap                  | 0.0225              | 0.0257             
r2                   | 0.5889              | 0.5029             
zpve                 | 0.3744              | 0.4365             
u0                   | 0.0750              | 0.0556             
u                    | 0.0848              | 0.0662             
h                    | 0.0848              | 0.0662             
g                    | 0.0616              | 0.0406             
cv                   | 0.4115              | 0

# Hypothesis 2

In [13]:
def manual_log_euclidean_vectorize(spd_matrices: np.ndarray) -> np.ndarray:
    """Manually computes the Log-Euclidean vectorization of a tensor of SPD matrices.
    
    Weights off-diagonal elements by sqrt(2) to preserve the Riemannian metric inner product.
    """
    N_molecules, d, _ = spd_matrices.shape
    triu_idx = np.triu_indices(d)
    
    # Generate an explicit weight matrix: 1.0 on diagonal, sqrt(2) on off-diagonals
    weight_matrix = np.where(np.eye(d, dtype=bool), 1.0, np.sqrt(2.0))
    
    vectorized_dataset = []
    
    for C in spd_matrices:
        # 1. Stable Matrix Logarithm for Symmetric Positive Definite matrices
        eigenvalues, eigenvectors = np.linalg.eigh(C)
        
        # Micro-safety clip to ensure numerical noise doesn't hit log(0)
        eigenvalues = np.clip(eigenvalues, a_min=1e-12, a_max=None)
        
        # Reconstruct the Matrix Logarithm
        log_C = eigenvectors @ np.diag(np.log(eigenvalues)) @ eigenvectors.T
        
        # 2. Scale elements to maintain inner-product isometry
        weighted_log_C = log_C * weight_matrix
        
        # 3. Flatten the upper triangle into a 1D vector
        flat_vector = weighted_log_C[triu_idx]
        vectorized_dataset.append(flat_vector)
        
    return np.array(vectorized_dataset)

In [19]:
import numpy as np
import polars as pl
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

def run_tuned_predictive_benchmark(df: pl.DataFrame, target_property: str = "cv", pca=False) -> None:
    print(f"=== Starting Tuned Predictive Benchmark for Target: {target_property} ===\n")
    
    # 1. Extract Riemann Tangent Space Vectors
    print("Extracting and vectorizing Riemannian covariance matrices...")
    spd_matrices = Riemann.get_spd_matrices(df, descriptor="soap", pca=pca)
    X_riemann = manual_log_euclidean_vectorize(spd_matrices)
    
    # 2. Extract Pre-averaged SOAP Embeddings from Polars
    print("Extracting pre-averaged SOAP embeddings...")
    # Handles list-type columns cleanly by stacking them into a 2D numpy array
    X_avg_soap = np.vstack(df.get_column("soap_embedding").to_list())
    
    # 3. Extract target property
    y = df.get_column(target_property).to_numpy()
    
    # 4. Train/Test Split (80/20) using indices to ensure perfect row alignment
    indices = np.arange(len(y))
    idx_train, idx_test = train_test_split(indices, test_size=0.2, random_state=42)
    
    # 5. Define Alpha Tuning Range (100 values from 10^-4 to 10^4)
    alphas_to_tune = np.logspace(-4, 4, 100)
    
    # =====================================================================
    # MODEL 1: Tuned Average SOAP Baseline
    # =====================================================================
    print("\nTuning and training Baseline Average SOAP model...")
    # RidgeCV automatically finds the best alpha using efficient built-in Cross-Validation
    ridge_soap = RidgeCV(alphas=alphas_to_tune, cv=5, scoring='neg_mean_absolute_error')
    ridge_soap.fit(X_avg_soap[idx_train], y[idx_train])
    
    preds_soap = ridge_soap.predict(X_avg_soap[idx_test])
    mae_soap = mean_absolute_error(y[idx_test], preds_soap)
    r2_soap = r2_score(y[idx_test], preds_soap)
    
    # =====================================================================
    # MODEL 2: Tuned Riemannian Covariance
    # =====================================================================
    print("Tuning and training Riemannian Tangent Space model...")
    ridge_riemann = RidgeCV(alphas=alphas_to_tune, cv=5, scoring='neg_mean_absolute_error')
    ridge_riemann.fit(X_riemann[idx_train], y[idx_train])
    
    preds_riemann = ridge_riemann.predict(X_riemann[idx_test])
    mae_riemann = mean_absolute_error(y[idx_test], preds_riemann)
    r2_riemann = r2_score(y[idx_test], preds_riemann)
    
    # =====================================================================
    # FINAL RESULTS DISPLAY
    # =====================================================================
    print("\n" + "="*60)
    print(f" BENCHMARK RESULTS FOR PERFORMANCE ON TEST SET ({len(idx_test)} Molecules)")
    print("="*60)
    
    print(f"{'Metric / Model':<25} | {'Average SOAP':<16} | {'Riemann Tangent':<16}")
    print("-" * 63)
    print(f"{'Optimal Alpha (α)':<25} | {ridge_soap.alpha_:<16.4f} | {ridge_riemann.alpha_:<16.4f}")
    print(f"{'Mean Absolute Error (MAE)':<25} | {mae_soap:<16.4f} | {mae_riemann:<16.4f}")
    print(f"{'R-squared (R²) Score':<25} | {r2_soap:<16.4f} | {r2_riemann:<16.4f}")
    print("="*60)
    
    # Quick comparative statement
    if mae_riemann < mae_soap:
        improvement = ((mae_soap - mae_riemann) / mae_soap) * 100
        print(f"🚀 Success! Riemann Tangent Space reduced error by {improvement:.2f}% over Average SOAP.")
    else:
        print("Average SOAP maintained lower error on this specific task configuration.")

# To run this pipeline, execute:
df_random = df.sample(1000)
run_tuned_predictive_benchmark(df_random, target_property="cv", pca=False)

2026-05-23 16:44:23.474 | INFO     | src.non_euclidean:_feature_matrices_from_df:332 - Using column: soap_matrix from df


=== Starting Tuned Predictive Benchmark for Target: cv ===

Extracting and vectorizing Riemannian covariance matrices...
Extracting pre-averaged SOAP embeddings...

Tuning and training Baseline Average SOAP model...
Tuning and training Riemannian Tangent Space model...

 BENCHMARK RESULTS FOR PERFORMANCE ON TEST SET (200 Molecules)
Metric / Model            | Average SOAP     | Riemann Tangent 
---------------------------------------------------------------
Optimal Alpha (α)         | 0.0001           | 4.0370          
Mean Absolute Error (MAE) | 1.3511           | 0.7748          
R-squared (R²) Score      | 0.7983           | 0.9335          
🚀 Success! Riemann Tangent Space reduced error by 42.65% over Average SOAP.
